# Chapter 4: Propositional Logic

```{admonition} Learning Objectives
:class: tip
- Understand propositional logic syntax and semantics
- Master logical operators and truth tables
- Convert formulas to Conjunctive Normal Form (CNF)
- Implement inference rules: Modus Ponens, Resolution
- Build knowledge bases for reasoning
- Apply forward and backward chaining
- Design rule-based expert systems
```

```{epigraph}
Logic is the beginning of wisdom, not the end.

-- Leonard Nimoy (as Spock)
```

## 4.1 Introduction

Logic is the foundation of knowledge representation and reasoning in AI. It provides a formal language for expressing facts and a calculus for deriving new facts.

### Why Logic in AI?

**Advantages:**
1. **Declarative**: Specify what is true, not how to compute it
2. **Compositional**: Build complex statements from simple ones
3. **Unambiguous**: Precise mathematical semantics
4. **Inference**: Derive new knowledge from existing knowledge
5. **Explainable**: Reasoning steps are transparent

**Applications:**
- Expert systems (medical diagnosis, financial planning)
- Hardware verification (chip design correctness)
- Software verification (program correctness)
- Planning and scheduling
- Natural language understanding
- Semantic web and knowledge graphs

In [ ]:
from typing import List, Set, Dict, Tuple, Optional
from dataclasses import dataclass
from abc import ABC, abstractmethod
from collections import defaultdict

print('✓ Libraries imported successfully')

## 4.2 Propositional Logic: The Basics

### Syntax

**Atomic Propositions**: Simple statements that are either true or false
- Example: P = "It is raining", Q = "I have an umbrella"
- Denoted by capital letters: P, Q, R, S, ...

**Logical Connectives:**

1. **Negation** (¬): NOT
2. **Conjunction** (∧): AND
3. **Disjunction** (∨): OR
4. **Implication** (→): IF...THEN (P → Q ≡ ¬P ∨ Q)
5. **Biconditional** (↔): IF AND ONLY IF

### Semantics

**Model**: Assignment of truth values to all propositions
**Validity**: Formula is true in all models (tautology)
**Satisfiability**: Formula is true in at least one model

In [ ]:
class Expr(ABC):
    """Base class for logical expressions."""
    
    @abstractmethod
    def evaluate(self, model: Dict[str, bool]) -> bool:
        pass
    
    @abstractmethod
    def get_symbols(self) -> Set[str]:
        pass
    
    @abstractmethod
    def __str__(self) -> str:
        pass

class Symbol(Expr):
    def __init__(self, name: str):
        self.name = name
    
    def evaluate(self, model: Dict[str, bool]) -> bool:
        return model.get(self.name, False)
    
    def get_symbols(self) -> Set[str]:
        return {self.name}
    
    def __str__(self) -> str:
        return self.name

class Not(Expr):
    def __init__(self, operand: Expr):
        self.operand = operand
    
    def evaluate(self, model: Dict[str, bool]) -> bool:
        return not self.operand.evaluate(model)
    
    def get_symbols(self) -> Set[str]:
        return self.operand.get_symbols()
    
    def __str__(self) -> str:
        return f'¬{self.operand}'

class And(Expr):
    def __init__(self, *operands: Expr):
        self.operands = list(operands)
    
    def evaluate(self, model: Dict[str, bool]) -> bool:
        return all(op.evaluate(model) for op in self.operands)
    
    def get_symbols(self) -> Set[str]:
        return set().union(*(op.get_symbols() for op in self.operands))
    
    def __str__(self) -> str:
        return '(' + ' ∧ '.join(str(op) for op in self.operands) + ')'

class Or(Expr):
    def __init__(self, *operands: Expr):
        self.operands = list(operands)
    
    def evaluate(self, model: Dict[str, bool]) -> bool:
        return any(op.evaluate(model) for op in self.operands)
    
    def get_symbols(self) -> Set[str]:
        return set().union(*(op.get_symbols() for op in self.operands))
    
    def __str__(self) -> str:
        return '(' + ' ∨ '.join(str(op) for op in self.operands) + ')'

class Implies(Expr):
    def __init__(self, antecedent: Expr, consequent: Expr):
        self.antecedent = antecedent
        self.consequent = consequent
    
    def evaluate(self, model: Dict[str, bool]) -> bool:
        return not self.antecedent.evaluate(model) or self.consequent.evaluate(model)
    
    def get_symbols(self) -> Set[str]:
        return self.antecedent.get_symbols() | self.consequent.get_symbols()
    
    def __str__(self) -> str:
        return f'({self.antecedent} → {self.consequent})'

print('✓ Logical expression classes defined')

## 4.3 Truth Tables

A truth table shows the value of a formula for all possible assignments.

**Example: P → Q**

| P | Q | P → Q |
|---|---|-------|
| T | T | T |
| T | F | F |
| F | T | T |
| F | F | T |

In [ ]:
def truth_table(expr: Expr, show_table: bool = True) -> List[Dict[str, bool]]:
    symbols = sorted(expr.get_symbols())
    n = len(symbols)
    
    if show_table:
        header = ' | '.join(symbols) + ' | ' + str(expr)
        print(header)
        print('-' * len(header))
    
    models = []
    for i in range(2 ** n):
        model = {}
        row = []
        
        for j, symbol in enumerate(symbols):
            value = bool((i >> (n - j - 1)) & 1)
            model[symbol] = value
            row.append('T' if value else 'F')
        
        result = expr.evaluate(model)
        row.append('T' if result else 'F')
        
        if show_table:
            print(' | '.join(row))
        
        if result:
            models.append(model.copy())
    
    return models

def is_valid(expr: Expr) -> bool:
    """Check if expression is valid (tautology)."""
    symbols = list(expr.get_symbols())
    n = len(symbols)
    
    for i in range(2 ** n):
        model = {}
        for j, symbol in enumerate(symbols):
            model[symbol] = bool((i >> (n - j - 1)) & 1)
        
        if not expr.evaluate(model):
            return False
    return True

print('✓ Truth table functions implemented')

In [ ]:
# Example: Truth table for implication
print('=== Truth Table Example: P → Q ===')
print()

P = Symbol('P')
Q = Symbol('Q')
impl = Implies(P, Q)

models = truth_table(impl)
print(f'\nSatisfying models: {len(models)}/4')
print()

# Check validity of law of excluded middle
print('=== Testing Validity: P ∨ ¬P ===')
excluded_middle = Or(P, Not(P))
print(f'Formula: {excluded_middle}')
print(f'Is valid (tautology)? {is_valid(excluded_middle)}')

## 4.4 Resolution

**Resolution Rule**:
```
P ∨ Q
¬P ∨ R
-------
Q ∨ R
```

Resolution is:
- **Sound**: Only derives true conclusions
- **Complete**: Can derive all entailed formulas (in CNF)
- **Refutation-complete**: Can prove KB ⊨ α by showing KB ∧ ¬α is unsatisfiable

In [ ]:
@dataclass
class Clause:
    """A clause in CNF (disjunction of literals)."""
    literals: Set[Tuple[str, bool]]
    
    def __init__(self, literals=None):
        if literals is None:
            self.literals = set()
        else:
            self.literals = set(literals)
    
    def is_empty(self) -> bool:
        return len(self.literals) == 0
    
    def __str__(self) -> str:
        if self.is_empty():
            return '⊥'
        lit_strs = []
        for symbol, is_pos in sorted(self.literals):
            lit_strs.append(symbol if is_pos else f'¬{symbol}')
        return ' ∨ '.join(lit_strs)
    
    def __hash__(self):
        return hash(frozenset(self.literals))
    
    def __eq__(self, other):
        return isinstance(other, Clause) and self.literals == other.literals

def resolve(clause1: Clause, clause2: Clause) -> Optional[Clause]:
    """Apply resolution rule to two clauses."""
    for lit1 in clause1.literals:
        symbol1, is_pos1 = lit1
        complement = (symbol1, not is_pos1)
        
        if complement in clause2.literals:
            resolvent_literals = (clause1.literals - {lit1}) | (clause2.literals - {complement})
            return Clause(resolvent_literals)
    return None

print('✓ Resolution inference implemented')

In [ ]:
# Test resolution
print('=== Resolution Example ===')
print()

# KB: P → Q, Q → R, P
# Query: R
kb = [
    Clause({('P', False), ('Q', True)}),  # ¬P ∨ Q
    Clause({('Q', False), ('R', True)}),  # ¬Q ∨ R
    Clause({('P', True)})                  # P
]

negated_query = [Clause({('R', False)})]  # ¬R

print('Knowledge Base:')
for clause in kb:
    print(f'  {clause}')
print(f'\nNegated Query: {negated_query[0]}')
print('\nResolving...')

# Simple resolution proof
clauses = set(kb + negated_query)
for c1 in kb:
    for c2 in kb:
        if c1 != c2:
            resolvent = resolve(c1, c2)
            if resolvent:
                print(f'{c1} + {c2} = {resolvent}')

## 4.5 Forward and Backward Chaining

Efficient inference for **definite clauses** (Horn clauses).

### Forward Chaining
- **Data-driven** (bottom-up)
- Start with facts, apply rules
- Derives all consequences

### Backward Chaining  
- **Goal-driven** (top-down)
- Start with query, find supporting facts
- More focused, efficient for specific queries

In [ ]:
class KnowledgeBase:
    """Knowledge base for definite clauses."""
    
    def __init__(self):
        self.facts = set()
        self.rules = []
    
    def tell_fact(self, fact: str):
        self.facts.add(fact)
    
    def tell_rule(self, premises: List[str], conclusion: str):
        self.rules.append((premises, conclusion))
    
    def forward_chain(self, query: str, verbose: bool = True) -> bool:
        """Forward chaining inference."""
        inferred = set(self.facts)
        agenda = list(self.facts)
        
        if verbose:
            print('Forward Chaining:')
            print(f'Initial facts: {sorted(inferred)}')
            print()
        
        while agenda:
            fact = agenda.pop(0)
            
            if fact == query:
                if verbose:
                    print(f'\n✓ Query \'{query}\' proved!')
                return True
            
            for premises, conclusion in self.rules:
                if conclusion not in inferred and all(p in inferred for p in premises):
                    inferred.add(conclusion)
                    agenda.append(conclusion)
                    if verbose:
                        print(f'Applied: {" ∧ ".join(premises)} → {conclusion}')
        
        if verbose:
            print(f'\n✗ Query \'{query}\' cannot be proved.')
        return False
    
    def backward_chain(self, query: str, visited: Set[str] = None, 
                        depth: int = 0, verbose: bool = True) -> bool:
        """Backward chaining inference."""
        if visited is None:
            visited = set()
            if verbose:
                print('Backward Chaining:')
                print(f'Goal: {query}')
                print()
        
        indent = '  ' * depth
        
        if query in self.facts:
            if verbose:
                print(f'{indent}✓ {query} is a known fact')
            return True
        
        if query in visited:
            return False
        
        visited.add(query)
        
        for premises, conclusion in self.rules:
            if conclusion == query:
                if verbose:
                    print(f'{indent}Trying rule: {" ∧ ".join(premises)} → {conclusion}')
                
                all_proved = True
                for premise in premises:
                    if not self.backward_chain(premise, visited, depth + 1, verbose):
                        all_proved = False
                        break
                
                if all_proved:
                    if verbose and depth == 0:
                        print(f'\n✓ Query \'{query}\' proved!')
                    return True
        
        if verbose and depth == 0:
            print(f'\n✗ Query \'{query}\' cannot be proved.')
        
        visited.remove(query)
        return False

print('✓ Knowledge Base implemented')

In [ ]:
# Example: Animal classification
print('=== Knowledge Base Example ===')
print()

kb = KnowledgeBase()

# Facts
kb.tell_fact('hasFur')
kb.tell_fact('produceMilk')

# Rules
kb.tell_rule(['hasFur', 'produceMilk'], 'mammal')
kb.tell_rule(['mammal', 'eatsMeat'], 'carnivore')

print('Knowledge Base:')
print('Facts:', sorted(kb.facts))
print('Rules:')
for premises, conclusion in kb.rules:
    print(f'  {" ∧ ".join(premises)} → {conclusion}')
print()

print('\n--- Testing Forward Chaining ---')
kb.forward_chain('mammal')
print()

print('\n--- Testing Backward Chaining ---')
kb.backward_chain('mammal')

## Programming Tasks

### Task 1: SAT Solver with DPLL (Hard)
Implement DPLL algorithm for SAT solving

### Task 2: Expert System (Medium)
Build a medical diagnosis expert system

### Task 3: Logic Puzzle Solver (Medium)
Solve logic puzzles (e.g., Einstein's riddle)

### Task 4: CNF Converter (Medium)
Convert arbitrary formulas to CNF

### Task 5: Backward Chaining (Easy-Medium)
Enhance backward chaining with memoization

## Summary

**Key Concepts**:
- Syntax: Propositions and connectives
- Semantics: Truth assignments and models
- CNF: Standard form for inference
- Resolution: Complete inference rule
- Forward/Backward chaining: Efficient for definite clauses

### Inference Methods

| Method | Complete | Complexity | Best For |
|--------|----------|------------|----------|
| Truth Tables | Yes | O(2^n) | Small formulas |
| Resolution | Yes | Exponential | Theorem proving |
| Forward Chain | Yes* | O(n) | Data-driven |
| Backward Chain | Yes* | O(n) | Goal-driven |

*For definite clauses

**Next**: First-Order Logic adds variables and quantifiers